In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchattacks
from torchattacks import FGSM, PGD, MIFGSM 
import pandas as pd
from functions import encode_variable
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn.functional as F
from pathlib import Path

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchattacks
from torchattacks import FGSM, PGD, MIFGSM 
import pandas as pd
from functions import encode_variable
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn.functional as F
from pathlib import Path

In [3]:
class RealTabularMIFGSM(torchattacks.MIFGSM):
    def __init__(self, model, eps=0.03, alpha=0.007, steps=10, decay=1.0, mask=None, feature_min = None, feature_max = None):
        super().__init__(model, eps, alpha, steps, decay)
        self.loss = nn.CrossEntropyLoss()  
        self.mask = mask 
        self.feature_min = feature_min
        self.feature_max = feature_max
        
    def forward(self, images, labels):
        """Overrides MIFGSM for tabular data."""
        
        adv_images = images.clone().detach()
        adv_images.requires_grad = True 
        
        momentum = 0
        for _ in range(self.steps):
            outputs = self.model(adv_images)
            loss = self.loss(outputs, labels)

            grad = torch.autograd.grad(loss, adv_images, retain_graph=True, create_graph=True, allow_unused=True)[0]

            if grad is None:
                print("ERROR: Gradient is None! adv_images is not in the graph!")
                break  

            grad = grad / torch.mean(torch.abs(grad), dim=(1,), keepdim=True)
            grad = grad + momentum * self.decay
            momentum = grad

            perturbation = self.alpha * grad.sign()
            if self.mask is not None:
                perturbation = perturbation * self.mask  # Zero out protected features

            adv_images = adv_images + perturbation
            adv_images = torch.clamp(adv_images, min=self.feature_min, max=self.feature_max) 
    
        return adv_images

In [4]:
class RealTabularPGD(torchattacks.PGD):
    def __init__(self, model, eps=0.03, alpha=0.007, steps=10, mask=None, feature_min=None, feature_max=None):
        super().__init__(model, eps, alpha, steps)
        self.loss = nn.CrossEntropyLoss()
        self.mask = mask
        self.feature_min = feature_min
        self.feature_max = feature_max
        
    def forward(self, images, labels):
        adv_images = images.clone().detach()
        adv_images.requires_grad = True
        
        original_images = images.clone().detach()
        
        for _ in range(self.steps):
            outputs = self.model(adv_images)
            loss = self.loss(outputs, labels)
            
            grad = torch.autograd.grad(loss, adv_images, retain_graph=True, create_graph=True, allow_unused=True)[0]
            if grad is None:
                print("ERROR: Gradient is None! adv_images is not in the graph!")
                break
                
            perturbation = self.alpha * grad.sign()
            if self.mask is not None:
                perturbation = perturbation * self.mask
            
            adv_images = adv_images + perturbation
            
    
            perturbation = torch.clamp(adv_images - original_images, min=-self.eps, max=self.eps)
            adv_images = original_images + perturbation
            
            adv_images = torch.clamp(adv_images, min=self.feature_min, max=self.feature_max)
            adv_images = adv_images.detach()
            adv_images.requires_grad = True
            
        return adv_images

In [5]:
class RealTabularDIM(torch.nn.Module):
    def __init__(self, model, eps=0.03, alpha=0.007, steps=10, diversity_prob=0.5, mask=None, feature_min=None, feature_max=None):
        super().__init__()
        self.model = model
        self.eps = eps
        self.alpha = alpha
        self.steps = steps
        self.diversity_prob = diversity_prob
        self.loss = nn.CrossEntropyLoss()
        self.mask = mask
        self.feature_min = feature_min
        self.feature_max = feature_max
    
    def diverse_input(self, x):
        # Randomly mask features with probability diversity_prob
        mask = (torch.rand_like(x) > self.diversity_prob).float()
        return x * mask + x * (1 - mask) * 0.1  # perturb masked features slightly
    
    def forward(self, images, labels):
        adv_images = images.clone().detach()
        adv_images.requires_grad = True
        
        original_images = images.clone().detach()
        
        for _ in range(self.steps):
            x_div = self.diverse_input(adv_images)
            outputs = self.model(x_div)
            loss = self.loss(outputs, labels)
            
            grad = torch.autograd.grad(loss, adv_images, retain_graph=True, create_graph=True, allow_unused=True)[0]
            if grad is None:
                print("ERROR: Gradient is None! adv_images is not in the graph!")
                break
                
            perturbation = self.alpha * grad.sign()
            if self.mask is not None:
                perturbation = perturbation * self.mask
            
            adv_images = adv_images + perturbation
            
            perturbation = torch.clamp(adv_images - original_images, min=-self.eps, max=self.eps)
            adv_images = original_images + perturbation
            
            adv_images = torch.clamp(adv_images, min=self.feature_min, max=self.feature_max)
            adv_images = adv_images.detach()
            adv_images.requires_grad = True
        
        return adv_images


In [6]:
class RealTabularTIM(torch.nn.Module):
    def __init__(self, model, eps=0.03, alpha=0.007, steps=10, mask=None, feature_min=None, feature_max=None):
        super().__init__()
        self.model = model
        self.eps = eps
        self.alpha = alpha
        self.steps = steps
        self.loss = nn.CrossEntropyLoss()
        self.mask = mask
        self.feature_min = feature_min
        self.feature_max = feature_max
        
        # smoothing kernel (e.g., [0.25, 0.5, 0.25])
        self.kernel = torch.tensor([0.25, 0.5, 0.25]).view(1, 1, -1)
        
    def smooth_grad(self, grad):
        grad = grad.unsqueeze(1)  # (batch, channel=1, features)
        grad_smoothed = F.conv1d(grad, self.kernel.to(grad.device), padding=1)
        return grad_smoothed.squeeze(1)
    
    def forward(self, images, labels):
        adv_images = images.clone().detach()
        adv_images.requires_grad = True
        
        original_images = images.clone().detach()
        
        for _ in range(self.steps):
            outputs = self.model(adv_images)
            loss = self.loss(outputs, labels)
            
            grad = torch.autograd.grad(loss, adv_images, retain_graph=True, create_graph=True, allow_unused=True)[0]
            if grad is None:
                print("ERROR: Gradient is None! adv_images is not in the graph!")
                break
            
            grad = self.smooth_grad(grad)
            
            perturbation = self.alpha * grad.sign()
            if self.mask is not None:
                perturbation = perturbation * self.mask
            
            adv_images = adv_images + perturbation
            
            perturbation = torch.clamp(adv_images - original_images, min=-self.eps, max=self.eps)
            adv_images = original_images + perturbation
            
            adv_images = torch.clamp(adv_images, min=self.feature_min, max=self.feature_max)
            adv_images = adv_images.detach()
            adv_images.requires_grad = True
        
        return adv_images

In [7]:
# Generating, training and testing surrogate model
class SurrogateModel(nn.Module):
    def __init__(self, input_dim):
        super(SurrogateModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128) 
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)  # Binary classification (Normal vs. Anomalous)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def test_surrogate(model_surrogate, x_test, y_test, device):
    model_surrogate.eval()  # Set the model to evaluation mode

    x_test_surrogate = torch.tensor(x_test, dtype=torch.float32).to(device)
    y_test_surrogate = torch.tensor(y_test, dtype=torch.long).to(device)

    with torch.no_grad():
        y_pred = model_surrogate(x_test_surrogate).argmax(axis=1).cpu().numpy()
    accuracy = accuracy_score(y_test_surrogate.cpu().numpy(), y_pred)
    from sklearn.metrics import recall_score
    recall = recall_score(y_test_surrogate.cpu().numpy(), y_pred)
    f1 = f1_score(y_test_surrogate.cpu().numpy(), y_pred, average='weighted')  # 'weighted' handles imbalanced classes
    asr = np.mean((y_test.cpu().numpy() == 1) & (y_pred == 0))
    original_results = {
        "Accuracy": accuracy, 
        "F1 Score": f1,
        "Recall": recall,
        "asr": asr,
    }
    print(f"Original results: {original_results}")
    return original_results

def train_surrogate(X_train, y_train, device):
    input_dim = X_train.shape[1]
    model_surrogate = SurrogateModel(input_dim).to(device)
    optimizer = optim.Adam(model_surrogate.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    for epoch in range(10):  # Train for a few epochs
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model_surrogate(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
    print("Surrogate model training complete.")
    return model_surrogate


In [8]:
def process_data(input_data, device):
    data = input_data.dropna(axis=0)
    data = data.loc[:, data.nunique() > 1]
    data.columns = [col.strip().lower() for col in data.columns]
    new_cols = ['timestamp' if 'starttimestamp' in col else col for col in data.columns]
    data.columns = new_cols
    data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
    data = data.sort_values(by='timestamp', ascending=True)
    features = [col for col in data.columns if col not in ['timestamp', 'anomaly', 'normal/attack',  'startdate', 'weekdaystart', 'yearstart', 'hourstart', 'minutestart', 'enddate', 'endtimestamp', 'weekdayend', 'yearend', 'hourend', 'minuteend', 'class']]
    print(features)
    if "connectortype" in features:
        data = encode_variable(data, data.columns.get_loc("connectortype"))
    for feature in features:
        data[feature] = pd.to_numeric(data[feature], errors='coerce')
    data.dropna()
    #data = data.sample(frac=1).reset_index(drop=True)
    x_data = data.loc[:, features].values
    y_data = data.loc[:, ['anomaly']].values.ravel()
    size = len(x_data)
    size_init = int(size * 0.8)
    x_train = x_data[:size_init]
    y_train = y_data[:size_init]
    x_test = x_data[size_init:]
    y_test = y_data[size_init:]
    df = pd.DataFrame(x_test, columns=features)
    df["anomaly"] = y_test
    df = df.sample(frac=1).reset_index(drop=True)
    x_test = df.loc[:, features].values
    y_test = df.loc[:, ['anomaly']].values.ravel()
    x_train = np.array(x_train, dtype=np.float32) 
    x_test = np.array(x_test, dtype=np.float32)
    x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)
    x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.long).to(device)

    return data, x_train, x_test, y_train, y_test, features
			
def load_anomalous_test_samples(x_test, y_test):
    anomalous_indices = np.where(y_test == 1)[0] 
    X_anomalous = x_test[anomalous_indices]
    y_anomalous = y_test[anomalous_indices]
    return X_anomalous, y_anomalous, anomalous_indices

def process_anomalous_samples(x_train, y_train, x_test, y_test, device):
    # Prepare anomalous x y for perturbation
    X_anomalous_test, y_anomalous_test, anomalous_indices_test = load_anomalous_test_samples(x_test, y_test)
    X_anomalous_train, y_anomalous_train, anomalous_indices_train = load_anomalous_test_samples(x_train, y_train)
    x_anom_train = torch.tensor(X_anomalous_train, dtype=torch.float32).clone().detach().requires_grad_(True).to(device)
    x_anom_train = torch.tensor(X_anomalous_train, dtype=torch.float32).to(device)
    x_anom_train.requires_grad = True
    y_anom_train = torch.tensor(y_anomalous_train, dtype=torch.long).to(device)
    y_anom_train = y_anom_train.view(-1).to(device)
    
    x_anom_test = torch.tensor(X_anomalous_test, dtype=torch.float32).clone().detach().requires_grad_(True).to(device)
    x_anom_test = torch.tensor(X_anomalous_test, dtype=torch.float32).to(device)
    x_anom_test.requires_grad = True
    y_anom_test = torch.tensor(y_anomalous_test, dtype=torch.long).to(device)
    y_anom_test = y_anom_test.view(-1).to(device)

    return x_anom_train, y_anom_train, x_anom_test, y_anom_test, anomalous_indices_test, anomalous_indices_train

def extract_column_stats(df):
    stats = {}
    feature_min = []
    feature_max = []
    for column in df.columns:
        stats[column] = {
            'max': df[column].max(),
            'min': df[column].min(),
            'mean': df[column].mean()
        }
        feature_min.append(df[column].min())
        feature_max.append(df[column].max())
    print(stats)
    print(feature_min)
    print(feature_max)
    return stats, feature_min, feature_max

def generateMask(x): 
    mask = []
    for i in range(x.cpu().numpy().shape[1]):  
        col = x.cpu().numpy()[:, i] 
        unique_values = np.unique(col)  
        if len(unique_values) < 20 or np.issubdtype(col.dtype, np.integer):
            mask.append(0) 
        else:
            mask.append(1)
    print(mask)
    return mask

def save_samples_csv(adversarial_samples, adv_dir, data, features, y):
    adversarial_dataset = pd.DataFrame(adversarial_samples.detach().cpu().numpy(), columns=data.loc[:, features].columns)
    adversarial_dataset['anomaly'] = y.detach().cpu().numpy()
    adversarial_dataset.to_csv(adv_dir, index=False)
    #print(f"{adv_dir} converted to CSV")

def performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha):
    mifgsm = RealTabularMIFGSM(model_surrogate, eps=eps, alpha=alpha, steps=steps, decay=0.8, mask=mask, feature_min=feature_min,
                                    feature_max=feature_max)
    mifgsm_samples = mifgsm(x, y)
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    Path(adv_dir).parent.mkdir(parents=True, exist_ok=True)
    save_samples_csv(mifgsm_samples, adv_dir, data, features, y)

    pgd = RealTabularPGD(model_surrogate, eps=eps, alpha=alpha, steps=steps, mask=mask, feature_min=feature_min, feature_max=feature_max)
    pgd_samples = pgd(x, y)
    adv_dir = f'./data/aexamples/{file}/pgd/{stage}/adversarialexamples_eps{str(eps)}.csv'
    Path(adv_dir).parent.mkdir(parents=True, exist_ok=True)
    save_samples_csv(pgd_samples, adv_dir, data, features, y)

    dim= RealTabularDIM(model_surrogate, eps=eps, alpha=alpha, steps=steps, diversity_prob=0.5, mask=mask, feature_min=feature_min, feature_max=feature_max)
    dim_samples = dim(x, y)
    adv_dir = f'./data/aexamples/{file}/dim/{stage}/adversarialexamples_eps{str(eps)}.csv'
    Path(adv_dir).parent.mkdir(parents=True, exist_ok=True)
    save_samples_csv(dim_samples, adv_dir, data, features, y)

    tim = RealTabularTIM(model_surrogate, eps=eps, alpha=alpha, steps=steps, mask=mask, feature_min=feature_min, feature_max=feature_max)
    tim_samples = tim(x, y)
    adv_dir = f'./data/aexamples/{file}/tim/{stage}/adversarialexamples_eps{str(eps)}.csv'
    Path(adv_dir).parent.mkdir(parents=True, exist_ok=True)
    save_samples_csv(tim_samples, adv_dir, data, features, y)

def attack(x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features):
    steps = 10
    epsilons = [0.0]
    ## First loop: from 0.003 to 0.03, step = 0.003
    #for value in np.arange(0.003, 0.031, 0.006): 
    #    eps = round(value, 3)
    #    alpha = round(eps/steps, 4)
    #    performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)
    #    epsilons.append(eps)

    # Second loop: from 0.03 to 0.3, step = 0.02
    for value in np.arange(0.01, 1.0, 0.1):
        eps = round(value, 3)
        alpha = round(eps/steps, 4)
        performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)
        epsilons.append(eps)
    if "train" in stage:
        for value in np.arange(0.003, 0.3, 0.05):
            eps = round(value, 3)
            alpha = round(eps/steps, 4)
            performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)

    return epsilons


In [9]:
def generate_ae_examples():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    data_path = Path('./data')
    original_results = {}
    files = [f for f in data_path.iterdir() if f.is_file()]
    for file in files:
        file_name = file.with_suffix('').as_posix().split('/')[1]
        original_results[file_name] = {}
        print(f"Processing file {file_name}")
        if 'xlsx' in file.name:
            data = pd.read_excel(file)
        else:
            data = pd.read_csv(file)
        data, x_train, x_test, y_train, y_test, features = process_data(data, device)
        print(np.unique(y_train.cpu().numpy(), return_counts=True))
        print(np.unique(y_test.cpu().numpy(), return_counts=True))
        # Train and test surrogate model
        model_surrogate = train_surrogate(x_train, y_train, device)
        original_results[file_name]['results'] = test_surrogate(model_surrogate, x_test, y_test, device)
        original_results[file_name]['x_test'] = x_test
        original_results[file_name]['y_test'] = y_test
        original_results[file_name]['x_train'] = x_train
        original_results[file_name]['y_train'] = y_train
        original_results[file_name]['features'] = features
        original_results[file_name]['model_surrogate'] = model_surrogate 
        # Generate data constraints
        stats, feature_min, feature_max = extract_column_stats(data.loc[:, features])
        mask = generateMask(x_train)
        feature_min = torch.tensor(np.array(feature_min), dtype=torch.float32).to(device)
        feature_max = torch.tensor(np.array(feature_max), dtype=torch.float32).to(device)  
        mask = torch.tensor(np.array(mask), dtype=torch.float32).to(device) 

        x_anom_train, y_anom_train, x_anom_test, y_anom_test, anomalous_indices_test, anomalous_indices_train = process_anomalous_samples(x_train.cpu(), y_train.cpu(), x_test.cpu(), y_test.cpu(), device)
        original_results[file_name]['anomalous_indices_test'] = anomalous_indices_test
        original_results[file_name]['anomalous_indices_train'] = anomalous_indices_train
        epsilons = attack(x_anom_train, y_anom_train, mask, feature_min, feature_max, model_surrogate, file_name, 'train', data, features)
        epsilons = attack(x_anom_test, y_anom_test, mask, feature_min, feature_max, model_surrogate, file_name, 'test', data, features)

    return original_results, epsilons, device
        
original_results, epsilons, device = generate_ae_examples()

Device: cuda
Processing file aias_test
['ldr1', 'led2', 'led3', 'temperature1', 'potentiometer2', 'gas1', 'led1', 'air_quality1', 'temperature2', 'potentiometer1', 'humidity1']
(array([0, 1]), array([2830,  529]))
(array([0, 1]), array([706, 134]))
Surrogate model training complete.
Original results: {'Accuracy': 0.9416666666666667, 'F1 Score': 0.9389462868501157, 'Recall': 0.7164179104477612, 'asr': np.float64(0.04523809523809524)}
{'ldr1': {'max': np.float64(81.19205298013244), 'min': np.float64(1.456953642384106), 'mean': np.float64(73.24288816794916)}, 'led2': {'max': np.float64(92.94117647058825), 'min': np.float64(0.0), 'mean': np.float64(87.03762333702232)}, 'led3': {'max': np.float64(80.7843137254902), 'min': np.float64(0.0), 'mean': np.float64(71.34051524872869)}, 'temperature1': {'max': np.float64(26.58445104010605), 'min': np.float64(23.95938769125513), 'mean': np.float64(24.328626760056924)}, 'potentiometer2': {'max': np.float64(91.015625), 'min': np.float64(0.0), 'mean': n

In [10]:
# Test examples
def read_adv_data(advPath):
    adversarial_data = pd.read_csv(advPath)
    features = [col for col in adversarial_data.columns if col not in ['timestamp', 'anomaly', 'normal/attack',  'startdate', 'weekdaystart', 'yearstart', 'hourstart', 'minutestart', 'enddate', 'endtimestamp', 'weekdayend', 'yearend', 'hourend', 'minuteend', 'class']]
    adversarial_x_data = adversarial_data.loc[:, features].values
    adversarial_y_data = adversarial_data.loc[:, ['anomaly']].values.ravel()
    return adversarial_x_data, adversarial_y_data, features
    
def test_adversarial_samples(advPath, x_test, y_test, model_surrogate, anomalous_indices, device):
    adversarial_x_data, adversarial_y_data, _ = read_adv_data(advPath)
    X_modified, Y_modified  = x_test.clone().cpu().numpy(), y_test.clone().cpu().numpy()
    X_modified[anomalous_indices] = adversarial_x_data
    Y_modified[anomalous_indices] = adversarial_y_data

    try:
        y_pred = model_surrogate.predict(X_modified)
    except: 
        model_surrogate.eval()
        tensor_X_modified = torch.tensor(np.array(X_modified, dtype=np.float32), dtype=torch.float32).to(device)
        with torch.no_grad():
            y_pred = model_surrogate(tensor_X_modified).argmax(axis=1).cpu().numpy()

    accuracy = accuracy_score(Y_modified, y_pred)
    recall = recall_score(Y_modified, y_pred)
    f1 = f1_score(Y_modified, y_pred)
    asr = np.mean((y_test.cpu().numpy() == 1) & (y_pred == 0))
    return  accuracy, f1, recall, asr

def initializeResults(original_results, file_name, epsilons):
    metrics = ["Accuracy", "F1 Score", "Recall", "EIR"]
    attacks = ["MIFGSM", "PGD", "TIM", "DIM"]
    multi_index = pd.MultiIndex.from_product([attacks, metrics], names=["Attack", "Metric"])
    df_results = pd.DataFrame(index=multi_index, columns=epsilons)
    df_results.loc[("MIFGSM", "Accuracy"), 0.0] = original_results[file_name]['results']["Accuracy"]
    df_results.loc[("MIFGSM", "F1 Score"), 0.0] = original_results[file_name]['results']["F1 Score"]
    df_results.loc[("MIFGSM", "Recall"), 0.0] = original_results[file_name]['results']["Recall"]
    df_results.loc[("MIFGSM", "asr"), 0.0] = original_results[file_name]['results']["asr"]
    df_results.loc[("MIFGSM", "EIR"), 0.0] = "-"
    df_results.loc[("PGD", "Accuracy"), 0.0] = original_results[file_name]['results']["Accuracy"]
    df_results.loc[("PGD", "F1 Score"), 0.0] = original_results[file_name]['results']["F1 Score"]
    df_results.loc[("PGD", "Recall"), 0.0] = original_results[file_name]['results']["Recall"]
    df_results.loc[("PGD", "asr"), 0.0] = original_results[file_name]['results']["asr"]
    df_results.loc[("PGD", "EIR"), 0.0] = "-"
    df_results.loc[("TIM", "Accuracy"), 0.0] = original_results[file_name]['results']["Accuracy"]
    df_results.loc[("TIM", "F1 Score"), 0.0] = original_results[file_name]['results']["F1 Score"]
    df_results.loc[("TIM", "Recall"), 0.0] = original_results[file_name]['results']["Recall"]
    df_results.loc[("TIM", "asr"), 0.0] = original_results[file_name]['results']["asr"]
    df_results.loc[("TIM", "EIR"), 0.0] = "-"
    df_results.loc[("DIM", "Accuracy"), 0.0] = original_results[file_name]['results']["Accuracy"]
    df_results.loc[("DIM", "F1 Score"), 0.0] = original_results[file_name]['results']["F1 Score"]
    df_results.loc[("DIM", "Recall"), 0.0] = original_results[file_name]['results']["Recall"]
    df_results.loc[("DIM", "asr"), 0.0] = original_results[file_name]['results']["asr"]
    df_results.loc[("DIM", "EIR"), 0.0] = "-"
    return df_results

def obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device):
    # MIFGSM
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results[file].loc[("MIFGSM", "Accuracy"), eps] = accuracy
    df_results[file].loc[("MIFGSM", "F1 Score"), eps] = f1
    df_results[file].loc[("MIFGSM", "Recall"), eps] = recall
    df_results[file].loc[("MIFGSM", "asr"), eps] = asr
    eir = 1 - (df_results[file].loc[("MIFGSM", "Recall"), eps] / max(original_results[file]['results']["Recall"], 1e-8))
    df_results[file].loc[("MIFGSM", "EIR"), eps] = eir * 100

    # PGD
    adv_dir = f'./data/aexamples/{file}/pgd/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results[file].loc[("PGD", "Accuracy"), eps] = accuracy
    df_results[file].loc[("PGD", "F1 Score"), eps] = f1
    df_results[file].loc[("PGD", "Recall"), eps] = recall
    df_results[file].loc[("PGD", "asr"), eps] = asr
    eir = 1 - (df_results[file].loc[("PGD", "Recall"), eps] / max(original_results[file]['results']["Recall"], 1e-8))
    df_results[file].loc[("PGD", "EIR"), eps] = eir * 100

    # TIM
    adv_dir = f'./data/aexamples/{file}/tim/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results[file].loc[("TIM", "Accuracy"), eps] = accuracy
    df_results[file].loc[("TIM", "F1 Score"), eps] = f1
    df_results[file].loc[("TIM", "Recall"), eps] = recall
    df_results[file].loc[("TIM", "asr"), eps] = asr
    eir = 1 - (df_results[file].loc[("TIM", "Recall"), eps] / max(original_results[file]['results']["Recall"], 1e-8))
    df_results[file].loc[("TIM", "EIR"), eps] = eir * 100

    # DIM
    adv_dir = f'./data/aexamples/{file}/dim/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results[file].loc[("DIM", "Accuracy"), eps] = accuracy
    df_results[file].loc[("DIM", "F1 Score"), eps] = f1
    df_results[file].loc[("DIM", "Recall"), eps] = recall
    df_results[file].loc[("DIM", "asr"), eps] = asr
    eir = 1 - (df_results[file].loc[("DIM", "Recall"), eps] / max(original_results[file]['results']["Recall"], 1e-8))
    df_results[file].loc[("DIM", "EIR"), eps] = eir * 100

    return df_results
def completeTest(original_results, epsilons, device):
    df_results = {}
    stage = 'test'
    
    for file in original_results:
        df_results[file] = initializeResults(original_results, file, epsilons)
        x_test, y_test = original_results[file]['x_test'], original_results[file]['y_test']
        anomalous_indices = original_results[file]['anomalous_indices_test']
        model_surrogate = original_results[file]['model_surrogate']
        #for value in np.arange(0.003, 0.031, 0.006): 
        #    eps = round(value, 3)
        #    df_results = obtain_adv_results(df_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)
        
        #for value in np.arange(0.03, 0.3, 0.06):
        for value in np.arange(0.01, 1.0, 0.1):
            eps = round(value, 3)
            df_results = obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)

    return df_results
df_results = completeTest(original_results, epsilons, device)

In [11]:
df_results['evcs']

0.00      0.01      0.11       0.21       0.31  \
Attack Metric                                                         
MIFGSM Accuracy  0.979937   0.97898  0.968757   0.958305   0.948618   
       F1 Score  0.979723  0.929894  0.892121   0.850638   0.809321   
       Recall    0.909677  0.903474  0.837221   0.769479     0.7067   
       EIR              -  0.681942  7.965085  15.411893  22.313148   
PGD    Accuracy  0.979937   0.97898  0.968795   0.958151   0.947469   
       F1 Score  0.979723  0.929894  0.892267   0.850007   0.804224   
       Recall    0.909677  0.903474  0.837469   0.768486   0.699256   
       EIR              -  0.681942  7.937807  15.521004  23.131478   
TIM    Accuracy  0.979937  0.979325  0.970518   0.960219   0.951336   
       F1 Score  0.979723  0.931122  0.898817   0.858466   0.821213   
       Recall    0.909677  0.905707  0.848635   0.781886   0.724318   
       EIR              -  0.436443  6.710311  14.048009  20.376432   
DIM    Accuracy  0.979937  0.979707  0.978597   0.976223   0.974539   
       F1 Score  0.979723  0.932484  0.928526   0.919964   0.913804   
       Recall    0.909677  0.908189  0.900993   0.885608    0.87469   
       EIR              -  0.163666  0.954719   2.645936   3.846154   
MIFGSM asr       0.013937  0.014894  0.025117   0.035569   0.045256   
PGD    asr       0.013937  0.014894  0.025078   0.035722   0.046405   
TIM    asr       0.013937  0.014549  0.023356   0.033655   0.042538   
DIM    asr       0.013937  0.014166  0.015277   0.017651   0.019335   

                      0.41       0.51       0.61       0.71       0.81  \
Attack Metric                                                            
MIFGSM Accuracy   0.939122   0.931197   0.924075    0.91906   0.916916   
       F1 Score   0.765832   0.727024   0.690011   0.662624   0.650564   
       Recall     0.645161   0.593797   0.547643   0.515136   0.501241   
       EIR       29.078014  34.724495  39.798145  43.371522  44.899073   
PGD    Accuracy   0.937093   0.928325   0.920744   0.915729   0.913355   
       F1 Score   0.756123   0.712354   0.671949   0.643793   0.630047   
       Recall      0.63201   0.575186   0.526055   0.493548   0.478164   
       EIR       30.523732  36.770322  42.171304  45.744681  47.435897   
TIM    Accuracy   0.942224   0.934107   0.926487   0.921357   0.917337   
       F1 Score   0.780381   0.741553   0.702786   0.675308    0.65295   
       Recall     0.665261   0.612655   0.563275   0.530025    0.50397   
       EIR       26.868522  32.651391  38.079651  41.734861  44.599018   
DIM    Accuracy   0.973581   0.972701   0.971016     0.9696   0.969523   
       F1 Score   0.910273   0.907004   0.900695   0.895334   0.895042   
       Recall     0.868486   0.862779   0.851861    0.84268   0.842184   
       EIR        4.528096   5.155483   6.355701   7.364975   7.419531   
MIFGSM asr        0.054752   0.062677   0.069799   0.074814   0.076958   
PGD    asr        0.056781   0.065549    0.07313   0.078145   0.080519   
TIM    asr         0.05165   0.059767   0.067386   0.072517   0.076537   
DIM    asr        0.020293   0.021173   0.022858   0.024274   0.024351   

                      0.91  
Attack Metric               
MIFGSM Accuracy   0.914695  
       F1 Score   0.637841  
       Recall     0.486849  
       EIR       46.481178  
PGD    Accuracy   0.911402  
       F1 Score    0.61853  
       Recall     0.465509  
       EIR       48.827059  
TIM    Accuracy   0.916265  
       F1 Score   0.646859  
       Recall     0.497022  
       EIR       45.362793  
DIM    Accuracy    0.96757  
       F1 Score   0.887561  
       Recall     0.829529  
       EIR        8.810693  
MIFGSM asr        0.079179  
PGD    asr        0.082472  
TIM    asr        0.077609  
DIM    asr        0.026304

In [12]:
df_results['aias_test']

0.00      0.01      0.11      0.21       0.31       0.41  \
Attack Metric                                                                   
MIFGSM Accuracy  0.941667  0.939286  0.934524  0.930952    0.92619   0.922619   
       F1 Score  0.938946  0.786611  0.765957      0.75    0.72807   0.711111   
       Recall    0.716418  0.701493  0.671642  0.649254   0.619403   0.597015   
       EIR              -  2.083333      6.25     9.375  13.541667  16.666667   
PGD    Accuracy  0.941667  0.939286  0.934524  0.930952    0.92619   0.922619   
       F1 Score  0.938946  0.786611  0.765957      0.75    0.72807   0.711111   
       Recall    0.716418  0.701493  0.671642  0.649254   0.619403   0.597015   
       EIR              -  2.083333      6.25     9.375  13.541667  16.666667   
TIM    Accuracy  0.941667  0.939286  0.934524  0.930952    0.92619   0.922619   
       F1 Score  0.938946  0.786611  0.765957      0.75    0.72807   0.711111   
       Recall    0.716418  0.701493  0.671642  0.649254   0.619403   0.597015   
       EIR              -  2.083333      6.25     9.375  13.541667  16.666667   
DIM    Accuracy  0.941667  0.941667  0.934524  0.933333   0.930952   0.927381   
       F1 Score  0.938946   0.79668  0.765957  0.760684       0.75   0.733624   
       Recall    0.716418  0.716418  0.671642  0.664179   0.649254   0.626866   
       EIR              -       0.0      6.25  7.291667      9.375       12.5   
MIFGSM asr       0.045238  0.047619  0.052381  0.055952   0.060714   0.064286   
PGD    asr       0.045238  0.047619  0.052381  0.055952   0.060714   0.064286   
TIM    asr       0.045238  0.047619  0.052381  0.055952   0.060714   0.064286   
DIM    asr       0.045238  0.045238  0.052381  0.053571   0.055952   0.059524   

                      0.51      0.61       0.71       0.81       0.91  
Attack Metric                                                          
MIFGSM Accuracy   0.922619  0.920238   0.920238   0.920238   0.917857  
       F1 Score   0.711111  0.699552   0.699552   0.699552   0.687783  
       Recall     0.597015   0.58209    0.58209    0.58209   0.567164  
       EIR       16.666667     18.75      18.75      18.75  20.833333  
PGD    Accuracy   0.922619  0.920238   0.920238   0.920238   0.917857  
       F1 Score   0.711111  0.699552   0.699552   0.699552   0.687783  
       Recall     0.597015   0.58209    0.58209    0.58209   0.567164  
       EIR       16.666667     18.75      18.75      18.75  20.833333  
TIM    Accuracy   0.922619  0.920238   0.920238   0.920238   0.917857  
       F1 Score   0.711111  0.699552   0.699552   0.699552   0.687783  
       Recall     0.597015   0.58209    0.58209    0.58209   0.567164  
       EIR       16.666667     18.75      18.75      18.75  20.833333  
DIM    Accuracy      0.925   0.92381   0.922619   0.922619   0.921429  
       F1 Score   0.722467  0.716814   0.711111   0.711111   0.705357  
       Recall      0.61194  0.604478   0.597015   0.597015   0.589552  
       EIR       14.583333    15.625  16.666667  16.666667  17.708333  
MIFGSM asr        0.064286  0.066667   0.066667   0.066667   0.069048  
PGD    asr        0.064286  0.066667   0.066667   0.066667   0.069048  
TIM    asr        0.064286  0.066667   0.066667   0.066667   0.069048  
DIM    asr        0.061905  0.063095   0.064286   0.064286   0.065476

In [30]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

class DeepANN(nn.Module):
    def __init__(self, input_size, output_size, lr=0.01):
        super(DeepANN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, output_size)  # Output layer
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)  # Dropout to prevent overfitting

        # Loss function & optimizer
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.parameters(), lr=lr)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.dropout(self.relu(self.fc3(x)))
        x = self.fc4(x)  # No activation (CrossEntropyLoss expects raw logits)
        return x

    def fit(self, X_train, y_train, epochs=10):
        """Trains the model."""
        for epoch in range(epochs):
            self.optimizer.zero_grad()
            outputs = self(X_train)
            loss = self.criterion(outputs, y_train)
            loss.backward()
            self.optimizer.step()
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

    def predict(self, X):
        x = torch.tensor(np.array(X, dtype=float), dtype=torch.float32).to(device)
        """Returns predicted class labels (0 or 1)."""
        with torch.no_grad():
            logits = self(x)
            predictions = torch.argmax(logits, axis=1)  # Convert logits to class labels
        return predictions.cpu().numpy()

    def predict_proba(self, X):
        x = torch.tensor(np.array(X, dtype=float), dtype=torch.float32).to(device)
        """Returns predicted probabilities for each class."""
        with torch.no_grad():
            logits = self(x)
            probabilities = torch.softmax(logits, axis=1)  # Apply softmax for probabilities
        return probabilities.cpu().numpy()
    
def trainModelsTransfer(x_train, y_train):
    models = []
    # 1. - Train Catboost        
    catboost_model = CatBoostClassifier()
    catboost_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy(), verbose=False)
    models.append(catboost_model)
    # 2 . - Train LGBM        
    lgb_model = LGBMClassifier ()
    lgb_model. fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(lgb_model)
    # 3. - Train MLP        
    mlp_model = MLPClassifier()
    mlp_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(mlp_model)
    # 4. - Train RF        
    rf_model = RandomForestClassifier()
    rf_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(rf_model)
    # 5. - Train  XGB        
    xgb_model = XGBClassifier()
    xgb_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(xgb_model)
    # 6. - Train DeepANN
    input_size = x_train.shape[1]
    output_size = 2  # Binary classification (0 or 1)
    deepannmodel = DeepANN(input_size, output_size).to(device)
    #t_x_train = torch.tensor(np.array(x_train, dtype=float), dtype=torch.float32)
    #t_y_train = torch.tensor(y_train, dtype=torch.long)
    deepannmodel.fit(x_train, y_train, epochs=35)
    models.append(deepannmodel)
    return models

def initializeResultsTransfer(epsilons, models, x_test, y_test):
    x_test = x_test.cpu().numpy()
    y_test = y_test.cpu().numpy()
    metrics = ["Accuracy", "F1 Score", "Recall", "EIR"]
    attacks = ["MIFGSM", "PGD", "TIM", "DIM"]
    models_names = ["CATBOOST", "LGBM", "MLP", "RF", "XGB", "DeepANN"]
    multi_index = pd.MultiIndex.from_product([models_names, attacks, metrics], names=["Models", "Attack", "Metric"])
    df_results = pd.DataFrame(index=multi_index, columns=epsilons)
    for idx, model_name in enumerate(models_names):
        y_pred = models[idx].predict(x_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        asr = np.mean((y_test == 1) & (y_pred == 0))
        for attack in attacks:
            df_results.loc[(model_name, attack, "Accuracy"), 0.0] = accuracy
            df_results.loc[(model_name, attack, "F1 Score"), 0.0] = f1
            df_results.loc[(model_name, attack, "Recall"), 0.0] = recall
            df_results.loc[(model_name, attack, "asr"), 0.0] = asr         
            df_results.loc[(model_name, attack, "EIR"), 0.0] = "-"            
    return df_results


def obtain_adv_results_transfer(df_results, original_results,  file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, model_name, device):
    # MIFGSM
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results.loc[(model_name, "MIFGSM", "Accuracy"), eps] = accuracy
    df_results.loc[(model_name,"MIFGSM", "F1 Score"), eps] = f1
    df_results.loc[(model_name,"MIFGSM", "Recall"), eps] = recall
    df_results.loc[(model_name,"MIFGSM", "asr"), eps] = asr
    eir = 1 - (df_results.loc[(model_name,"MIFGSM", "Recall"), eps] / df_results.loc[(model_name,"MIFGSM", "Recall"), 0.0])
    df_results.loc[(model_name,"MIFGSM", "EIR"), eps] = eir * 100
    # PGD
    adv_dir = f'./data/aexamples/{file}/pgd/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results.loc[(model_name,"PGD", "Accuracy"), eps] = accuracy
    df_results.loc[(model_name,"PGD", "F1 Score"), eps] = f1
    df_results.loc[(model_name,"PGD", "Recall"), eps] = recall
    df_results.loc[(model_name,"PGD", "asr"), eps] = asr
    eir = 1 - (df_results.loc[(model_name,"PGD", "Recall"), eps] / df_results.loc[(model_name,"MIFGSM", "Recall"), 0.0])
    df_results.loc[(model_name,"PGD", "EIR"), eps] = eir * 100

    # TIM
    adv_dir = f'./data/aexamples/{file}/tim/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results.loc[(model_name,"TIM", "Accuracy"), eps] = accuracy
    df_results.loc[(model_name,"TIM", "F1 Score"), eps] = f1
    df_results.loc[(model_name,"TIM", "Recall"), eps] = recall
    df_results.loc[(model_name,"TIM", "asr"), eps] = asr
    eir = 1 - (df_results.loc[(model_name,"TIM", "Recall"), eps] / df_results.loc[(model_name,"MIFGSM", "Recall"), 0.0])
    df_results.loc[(model_name,"TIM", "EIR"), eps] = eir * 100

    # DIM
    adv_dir = f'./data/aexamples/{file}/dim/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall, asr = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results.loc[(model_name,"DIM", "Accuracy"), eps] = accuracy
    df_results.loc[(model_name,"DIM", "F1 Score"), eps] = f1
    df_results.loc[(model_name,"DIM", "Recall"), eps] = recall
    df_results.loc[(model_name,"DIM", "asr"), eps] = asr
    eir = 1 - (df_results.loc[(model_name,"DIM", "Recall"), eps] / df_results.loc[(model_name,"MIFGSM", "Recall"), 0.0])
    df_results.loc[(model_name,"DIM", "EIR"), eps] = eir * 100

    return df_results
def testTransferability(original_results, epsilons):
    transferability_results = {}
    models_names = ["CATBOOST", "LGBM", "MLP", "RF", "XGB", "DeepANN"]
    # Test normal
    for file in original_results:
        transferability_results[file] = {}
        x_train = original_results[file]['x_train']
        y_train = original_results[file]['y_train']
        x_test = original_results[file]['x_test']
        y_test = original_results[file]['y_test']
        models = trainModelsTransfer(x_train, y_train)
        df_results = initializeResultsTransfer(epsilons, models, x_test, y_test)
        transferability_results[file]['models'] = models
        transferability_results[file]['transfer_results'] = df_results
        for idx, model_name in enumerate(models_names):
            for value in np.arange(0.01, 1.0, 0.1):
                eps = round(value, 3)
                transferability_results[file]['transfer_results'] = obtain_adv_results_transfer(transferability_results[file]['transfer_results'] , original_results  , file, 'test',  x_test, y_test, original_results[file]['anomalous_indices_test'], eps, models[idx], model_name, device)

# df_results = completeTest(original_results, epsilons, device)
    return transferability_results

In [31]:
transferability_results = testTransferability(original_results, epsilons)

[LightGBM] [Info] Number of positive: 529, number of negative: 2830
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000342 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 439
[LightGBM] [Info] Number of data points in the train set: 3359, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.157487 -> initscore=-1.677044
[LightGBM] [Info] Start training from score -1.677044
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Epoch [1/35], Loss: 8.1042
Epoch [2/35], Loss: 10.8619
Epoch [3/35], Loss: 2.0756
Epoch [4/35], Loss: 23.9501
Epoch [5/35], Loss: 1.9342
Epoch [6/35], Loss: 3.4804
Epoch [7/35], Loss: 4.3066
Epoch [8/35], Loss: 3.6754
Epoch [9/35], Loss: 2.4805
Epoch [10/35], Loss: 1.2759
Epoch [11/35], Loss: 1.3288
Epoch [12/35], Loss: 0.6954
Epoch [13/35], Loss: 0.5088
Epoch [14/35], Loss: 0.5477
Epoch [15/35], Loss: 0.5245
Epoch [16/35], Loss: 0.

In [32]:
transferability_results['evcs']['transfer_results']

0.00      0.01      0.11       0.21       0.31  \
Models   Attack Metric                                                         
CATBOOST MIFGSM Accuracy   0.98189  0.981124  0.970136   0.956543   0.946206   
                F1 Score  0.940089  0.937397  0.897206    0.84308   0.798219   
                Recall    0.920844  0.915881  0.844665   0.756576   0.689578   
                EIR              -  0.538938  8.272703  17.838857  25.114524   
         PGD    Accuracy   0.98189  0.981124  0.970097   0.956543   0.945938   
...                            ...       ...       ...        ...        ...   
XGB      DIM    asr       0.014128  0.014473  0.015583   0.017804   0.020943   
DeepANN  MIFGSM asr       0.051114  0.051535  0.056857   0.062103    0.06685   
         PGD    asr       0.051114  0.051918  0.056934   0.062792    0.06685   
         TIM    asr       0.051114  0.052225  0.057087   0.061375   0.066544   
         DIM    asr       0.051114  0.051344  0.052875   0.054254   0.055594   

                               0.41       0.51       0.61       0.71  \
Models   Attack Metric                                                 
CATBOOST MIFGSM Accuracy   0.936672   0.929321   0.923348   0.919787   
                F1 Score   0.753649   0.716958   0.685517   0.666029   
                Recall     0.627792   0.580149   0.541439   0.518362   
                EIR       31.824306  36.998114  41.201832  43.707895   
         PGD    Accuracy   0.936595   0.928938   0.923884    0.92017   
...                             ...        ...        ...        ...   
XGB      DIM    asr        0.021594   0.025078    0.02661    0.02818   
DeepANN  MIFGSM asr        0.071407   0.075044   0.077801   0.080519   
         PGD    asr        0.071024   0.074508   0.077992   0.079639   
         TIM    asr        0.069416   0.072708   0.076537   0.078107   
         DIM    asr        0.056398   0.057776   0.059078   0.059691   

                               0.81       0.91  
Models   Attack Metric                          
CATBOOST MIFGSM Accuracy   0.918945    0.91703  
                F1 Score   0.661334    0.65054  
                Recall     0.512903   0.500496  
                EIR       44.300728  45.648073  
         PGD    Accuracy   0.918524   0.918332  
...                             ...        ...  
XGB      DIM    asr        0.028869   0.031549  
DeepANN  MIFGSM asr        0.082702   0.083889  
         PGD    asr        0.081591   0.082204  
         TIM    asr        0.079371   0.081285  
         DIM    asr        0.059767   0.059269  

[120 rows x 11 columns]

In [33]:
transferability_results['aias_test']['transfer_results']

0.00      0.01      0.11      0.21      0.31  \
Models   Attack Metric                                                       
CATBOOST MIFGSM Accuracy  0.995238  0.995238  0.989286  0.985714  0.980952   
                F1 Score  0.984962  0.984962  0.965517  0.953488  0.937008   
                Recall    0.977612  0.977612  0.940299   0.91791   0.88806   
                EIR              -       0.0  3.816794   6.10687  9.160305   
         PGD    Accuracy  0.995238  0.995238  0.989286  0.985714  0.980952   
...                            ...       ...       ...       ...       ...   
XGB      DIM    asr       0.004762  0.009524  0.015476  0.017857   0.02381   
DeepANN  MIFGSM asr            0.1   0.09881       0.1   0.09881  0.102381   
         PGD    asr            0.1   0.09881   0.09881  0.102381       0.1   
         TIM    asr            0.1       0.1       0.1   0.09881       0.1   
         DIM    asr            0.1   0.09881       0.1   0.09881       0.1   

                              0.41       0.51       0.61       0.71  \
Models   Attack Metric                                                
CATBOOST MIFGSM Accuracy   0.97619      0.975   0.972619   0.971429   
                F1 Score      0.92   0.915663   0.906883   0.902439   
                Recall    0.858209   0.850746   0.835821   0.828358   
                EIR       12.21374  12.977099  14.503817  15.267176   
         PGD    Accuracy   0.97619      0.975   0.972619   0.971429   
...                            ...        ...        ...        ...   
XGB      DIM    asr       0.027381   0.030952   0.030952   0.033333   
DeepANN  MIFGSM asr            0.1    0.09881        0.1        0.1   
         PGD    asr            0.1    0.09881    0.09881   0.102381   
         TIM    asr        0.10119    0.09881        0.1    0.09881   
         DIM    asr       0.102381    0.09881    0.09881        0.1   

                               0.81       0.91  
Models   Attack Metric                          
CATBOOST MIFGSM Accuracy   0.970238   0.963095  
                F1 Score   0.897959   0.870293  
                Recall     0.820896   0.776119  
                EIR       16.030534  20.610687  
         PGD    Accuracy   0.970238   0.964286  
...                             ...        ...  
XGB      DIM    asr        0.036905   0.042857  
DeepANN  MIFGSM asr         0.09881        0.1  
         PGD    asr         0.10119    0.10119  
         TIM    asr         0.10119        0.1  
         DIM    asr             0.1        0.1  

[120 rows x 11 columns]

In [34]:
transferability_results['evcs']['transfer_results'].to_csv('transfer_results_evcs.csv', index=True) 
transferability_results['aias_test']['transfer_results'].to_csv('transfer_results_aias.csv', index=True) 

In [35]:
## Testing models
from joblib import load
catboost_fgsmdim_model = load('./data/models/catboost_mifgsm_evcs.joblib')
keras_pgdtim_model = load('./data/models/keras_pgdtim_evcs.joblib')
models =[catboost_fgsmdim_model, keras_pgdtim_model] 

FileNotFoundError: [Errno 2] No such file or directory: './data/models/keras_pgdtim_evcs.joblib'

In [ ]:
data = pd.read_excel("./data/evcs.xlsx")
data = data.sort_values(by='startTimestamp', ascending=True)
encode_variable(data, data.columns.get_loc("connectorType"))

#features = ["connectorType", "durationSession", "durationCharge", "energy", 'cost', 'tariff', "meanPower", 'maxPower']
features = ['connectorType', 'durationCharge', 'durationSession', 'energy', 'tariff', 'cost', 'meanPower', 'maxPower']
x_evcs = data.loc[:, features].values
y_evcs = data.loc[:, ['anomaly']].values.ravel()
size = len(x_evcs)
size_init = int(size * 0.8)
x_init = x_evcs[:size_init]
y_init = y_evcs[:size_init]
x_inc = x_evcs[size_init:]
y_inc = y_evcs[size_init:]
df = pd.DataFrame(x_inc, columns=features)
df["anomaly"] = y_inc
df = df.sample(frac=1).reset_index(drop=True)
x_inc = df.loc[:, features].values
y_inc = df.loc[:, ['anomaly']].values.ravel()
y_det = np.zeros(len(y_inc))

In [ ]:
for det_model in models:
    print(det_model)
    y_pred_det = det_model.predict(x_inc)
    accuracy_det = accuracy_score(y_det, y_pred_det)
    recall_det = recall_score(y_det, y_pred_det)
    f1_det = f1_score(y_det, y_pred_det)

    print(f"Accuracy: {accuracy_det}")
    print(f"Recall: {recall_det}")
    print(f"f1-score: {f1_det}")

Accuracy: 0.8429435638257141
Recall: 0.0
f1-score: 0.0
KerasClassifier(
	model=<function build_model at 0x7f4ba35693a0>
	build_fn=None
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=32
	validation_batch_size=None
	verbose=0
	callbacks=None
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=10
	model__input_shape=(8,)
	class_weight=None
)
Accuracy: 0.8160272608928708
Recall: 0.0
f1-score: 0.0


In [ ]:
import os
import tensorflow as tf
import numpy as np

#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Force CPU

model = tf.keras.models.load_model("./data/models/keras_pgdtim_evcs.keras")

X = np.random.rand(1, 8).astype("float32")  # Adjust shape to match your model

print("Input shape:", X.shape)
print("Model input shape:", model.input_shape)

preds = model.predict(X)
print("Predictions:", preds)

Input shape: (1, 8)
Model input shape: (None, 8)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
Predictions: [[0.01514882]]
